# Libraries

In [5]:
# Run this cell first to install all required packages
!pip install pystac-client planetary-computer stackstac pyproj rioxarray dask odc-stac -q
!pip install rasterio python-dotenv requests segmentation-models-pytorch

# The Sen1Floods11 Collection Pipeline

In [2]:
import os
import subprocess

def setup_sen1floods11_pipeline(base_dir="./data/sen1floods11"):
    os.makedirs(base_dir, exist_ok=True)
   
    # Verified root path (from your gsutil ls)
    gs_hand_root = "gs://sen1floods11/v1.1/data/flood_events/HandLabeled/"
   
    # CORRECT layers for v1.1 (S1Hand / S2Hand / LabelHand)
    layers = ["S1Hand", "S2Hand", "LabelHand"]
   
    print(f"--- Initiating Sen1Floods11 HandLabeled Download to {base_dir} ---")
   
    for layer in layers:
        local_layer_path = os.path.join(base_dir, layer)
        os.makedirs(local_layer_path, exist_ok=True)
       
        print(f"Fetching Layer: {layer}...")
       
        cmd = f"gsutil -m cp -r -n {gs_hand_root}{layer} {base_dir}"
       
        try:
            subprocess.run(cmd, shell=True, check=True)
            print(f"✅ Successfully synced {layer}.")
        except subprocess.CalledProcessError as e:
            print(f"❌ Failed to sync {layer}. Running verification...")
            subprocess.run(f"gsutil ls {gs_hand_root}", shell=True)
    
    print("\n--- Final Integrity Check ---")
    for layer in layers:
        path = os.path.join(base_dir, layer)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"{layer}: {count} chips ready.")

if __name__ == "__main__":
    setup_sen1floods11_pipeline()

--- Initiating Sen1Floods11 HandLabeled Download to ./data/sen1floods11 ---
Fetching Layer: S1Hand...


Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_129334_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_294583_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_195474_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Ghana_1033830_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_290290_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_312675_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_379434_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_432776_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_314919_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_23014_S1Hand.tif...
Copying gs://sen1floods11/v1.1/d

✅ Successfully synced S1Hand.
Fetching Layer: S2Hand...


Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_242570_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_312675_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_360519_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_290290_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_379434_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_23014_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_195474_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_233925_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_314919_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Ghana_1033830_S2Hand.tif...
Copying gs://sen1floods11/v1.1/d

✅ Successfully synced S2Hand.
Fetching Layer: LabelHand...


Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_103757_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_129334_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_195474_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_233925_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_23014_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_290290_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_379434_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Ghana_103272_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Ghana_1033830_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivi

✅ Successfully synced LabelHand.

--- Final Integrity Check ---
S1Hand: 446 chips ready.
S2Hand: 446 chips ready.
LabelHand: 446 chips ready.


# Calculate NDWI/SAR indices and slice massive regions into ML-ready 512x512 tiles.

In [1]:
import os
import glob
import numpy as np
import rasterio
import pandas as pd
from tqdm import tqdm

def build_sen1floods_tensors(base_dir="./data/sen1floods11", output_dir="./data/ml_dataset"):
    """
    Phase 2: Ingests Sen1Floods11 Triplets, engineers features, 
    and outputs standardized 8-channel PyTorch tensors.
    """
    s1_dir = os.path.join(base_dir, 'S1Hand')
    s2_dir = os.path.join(base_dir, 'S2Hand')
    lbl_dir = os.path.join(base_dir, 'LabelHand')

    feat_dir = os.path.join(output_dir, 'features')
    mask_dir = os.path.join(output_dir, 'masks')
    os.makedirs(feat_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)

    s1_files = glob.glob(os.path.join(s1_dir, '*.tif'))
    if not s1_files:
        print(f"Error: No S1 files found in {s1_dir}")
        return

    manifest = []
    
    print(f"Engineering features and stacking {len(s1_files)} triplets...")

    for s1_path in tqdm(s1_files, desc="Processing Tensors"):
        filename = os.path.basename(s1_path)
        base_name = filename.replace('_S1Hand.tif', '')

        s2_path = os.path.join(s2_dir, f"{base_name}_S2Hand.tif")
        lbl_path = os.path.join(lbl_dir, f"{base_name}_LabelHand.tif")

        # Skip if the triplet is incomplete
        if not (os.path.exists(s2_path) and os.path.exists(lbl_path)):
            continue

        # Load Data
        with rasterio.open(s1_path) as src:
            s1_data = src.read() # Shape: [2, 512, 512]
        with rasterio.open(s2_path) as src:
            s2_data = src.read() # Shape: [13, 512, 512]
        with rasterio.open(lbl_path) as src:
            lbl_data = src.read(1) # Shape: [512, 512]

        # --- 1. SAR CALIBRATION ---
        # Convert linear backscatter to Decibels and scale to [0, 1]
        s1_valid = np.where(s1_data > 0, s1_data, 1e-7)
        s1_db = 10 * np.log10(s1_valid)
        s1_norm = np.clip((s1_db + 30) / 30.0, 0, 1)
        
        copol, crosspol = s1_norm[0], s1_norm[1]

        # --- 2. OPTICAL NORMALIZATION ---
        # Sen1Floods11 Optical Bands: 1=Blue, 2=Green, 3=Red, 7=NIR
        blue = np.clip(s2_data[1] / 10000.0, 0, 1)
        green = np.clip(s2_data[2] / 10000.0, 0, 1)
        red = np.clip(s2_data[3] / 10000.0, 0, 1)
        nir = np.clip(s2_data[7] / 10000.0, 0, 1)

        # --- 3. ENGINEERED INDICES ---
        with np.errstate(divide='ignore', invalid='ignore'):
            # NDWI = (Green - NIR) / (Green + NIR)
            ndwi = np.where((green + nir) > 0, (green - nir) / (green + nir), 0.0)
            
            # SAR Ratio = VV / VH
            sar_ratio = np.where(crosspol > 0, copol / crosspol, 0.0)

        # Sanitize math artifacts
        ndwi = np.nan_to_num(ndwi, nan=0.0, posinf=1.0, neginf=-1.0)
        sar_ratio = np.nan_to_num(sar_ratio, nan=0.0, posinf=0.0, neginf=0.0)

        # --- 4. TENSOR STACKING ---
        # Enforce strict 8-channel layout
        features = np.stack([copol, crosspol, blue, green, red, nir, ndwi, sar_ratio], axis=0).astype(np.float32)
        mask = lbl_data.astype(np.int8)

        # --- 5. EXPORT ---
        patch_id = f"SEN1_{base_name}"
        np.save(os.path.join(feat_dir, f"{patch_id}.npy"), features)
        np.save(os.path.join(mask_dir, f"{patch_id}.npy"), mask)

        # --- 6. MANIFEST METRICS ---
        # Calculate water ratio ignoring the -1 cloud pixels
        valid_pixels = np.sum(mask != -1)
        water_pixels = np.sum(mask == 1)
        water_ratio = (water_pixels / valid_pixels) if valid_pixels > 0 else 0.0

        manifest.append({
            "patch_id": patch_id,
            "biome": base_name.split('_')[0], # Captures region (e.g., 'Bolivia', 'Pakistan')
            "water_ratio": round(water_ratio, 4),
            "valid_pixel_ratio": round(valid_pixels / (512 * 512), 4)
        })

    # Save tracking manifest
    df = pd.DataFrame(manifest)
    df.to_csv(os.path.join(output_dir, 'patch_manifest.csv'), index=False)
    
    print("\n=== Phase 2 Complete ===")
    print(f"Successfully serialized {len(df)} 8-channel tensors.")
    print(f"Patches containing water: {len(df[df['water_ratio'] > 0.01])}")
    print(f"Dataset footprint saved to: {output_dir}")

# Execution
if __name__ == "__main__":
    build_sen1floods_tensors()

Engineering features and stacking 446 triplets...


Processing Tensors: 100%|██████████| 446/446 [00:21<00:00, 20.68it/s]


=== Phase 2 Complete ===
Successfully serialized 446 8-channel tensors.
Patches containing water: 293
Dataset footprint saved to: ./data/ml_dataset


# Try 2 - Model 4


## Step 0 — Download Official Sen1Floods11 Split CSVs

Run this cell **once** before training.
These CSVs define the exact patches used in all published Sen1Floods11 papers.
Using them makes your results directly comparable to Bonafilia 2020, Yadav 2022, etc.

If gsutil is unavailable (e.g. no GCS access), the training cell below will
automatically fall back to the custom seed=42 split.

In [2]:
import subprocess, os

OFFICIAL_TRAIN_CSV = "./flood_train_data.csv"
OFFICIAL_VAL_CSV   = "./flood_valid_data.csv"  # Correct name with 'i'
OFFICIAL_TEST_CSV  = "./flood_test_data.csv"
GCS_BASE = "gs://sen1floods11/v1.1/splits/flood_handlabeled"

def download_official_splits():
    # Primary file mappings with correct names
    files = {
        OFFICIAL_TRAIN_CSV: f"{GCS_BASE}/flood_train_data.csv",
        OFFICIAL_VAL_CSV  : f"{GCS_BASE}/flood_valid_data.csv",   # Note: valid (with i) not val
        OFFICIAL_TEST_CSV : f"{GCS_BASE}/flood_test_data.csv",
    }
    
    # Alternative names to try if primary fails
    alternative_names = {
        OFFICIAL_VAL_CSV: [
            f"{GCS_BASE}/flood_val_data.csv",      # Try val if valid doesn't work
            f"{GCS_BASE}/validation_data.csv",
            f"{GCS_BASE}/valid_data.csv"
        ],
        OFFICIAL_TEST_CSV: [
            f"{GCS_BASE}/test_data.csv",
            f"{GCS_BASE}/flood_test.csv"
        ]
    }
    
    all_ok = True
    
    for local, remote in files.items():
        if os.path.exists(local):
            print(f"  Already exists: {local}")
            continue
            
        print(f"  Downloading {remote} ...")
        result = subprocess.run(
            ["gsutil", "cp", remote, local],
            capture_output=True, text=True
        )
        
        if result.returncode == 0:
            print(f"  OK  Saved: {local}")
        else:
            # Try alternative names if primary fails
            success = False
            if local in alternative_names:
                for alt_remote in alternative_names[local]:
                    print(f"  Trying alternative: {alt_remote} ...")
                    alt_result = subprocess.run(
                        ["gsutil", "cp", alt_remote, local],
                        capture_output=True, text=True
                    )
                    if alt_result.returncode == 0:
                        print(f"  OK  Saved: {local} (from alternative source)")
                        success = True
                        break
            
            if not success:
                print(f"  FAIL: {result.stderr.strip()}")
                all_ok = False
    
    return all_ok

def list_available_files():
    """List all available files in the GCS bucket to see what's actually there"""
    print("\nChecking available files in GCS bucket...")
    result = subprocess.run(
        ["gsutil", "ls", f"{GCS_BASE}/"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("Available files:")
        for line in result.stdout.strip().split('\n'):
            if line:
                print(f"  {line}")
    else:
        print(f"Could not list directory: {result.stderr}")

# First try to download with the correct names
ok = download_official_splits()

if ok:
    print("\n✅ OK  Official splits ready.")
    print("    Training will use benchmark-comparable splits.")
else:
    print("\n⚠️ WARN Download failed or gsutil not available.")
    print("   Training will auto-fallback to custom seed=42 split.")
    print("   Results will NOT be directly comparable to published papers.")
    
    # Optional: List what's actually available
    list_available_files()
    
    # Create a validation split from training data as fallback
    print("\nCreating custom validation split from training data...")
    try:
        import pandas as pd
        from sklearn.model_selection import train_test_split
        
        if os.path.exists(OFFICIAL_TRAIN_CSV):
            train_data = pd.read_csv(OFFICIAL_TRAIN_CSV)
            train_df, val_df = train_test_split(
                train_data, 
                test_size=0.2, 
                random_state=42,
                stratify=train_data['label'] if 'label' in train_data.columns else None
            )
            
            # Save the validation data with the expected name
            val_df.to_csv(OFFICIAL_VAL_CSV, index=False)
            print(f"  Created validation set with {len(val_df)} samples")
            print(f"  Saved to: {OFFICIAL_VAL_CSV}")
            ok = True
        else:
            print("  Training data not found either!")
    except Exception as e:
        print(f"  Error creating fallback split: {e}")

  OK  Saved: ./flood_train_data.csv
  OK  Saved: ./flood_valid_data.csv
  OK  Saved: ./flood_test_data.csv

✅ OK  Official splits ready.
    Training will use benchmark-comparable splits.


## MANZAR v5 — Training with Official Split Auto-Detection

`make_splits()` now checks for official CSVs first.
- If `flood_train_data.csv` + `flood_val_data.csv` exist  →  official split used
- Otherwise  →  custom seed=42 fallback (same as v4)

In [6]:
import os, gc, csv, random, shutil, logging, json
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import KFold
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────
MANIFEST_PATH     = "./data/ml_dataset/patch_manifest.csv"
FEATURES_DIR      = "./data/ml_dataset/features"
MASKS_DIR         = "./data/ml_dataset/masks"
CHECKPOINT_DIR    = "./checkpoints"
LOG_CSV           = "./checkpoints/train_log.csv"
BEST_WEIGHTS      = "./checkpoints/best_val_iou.pth"
BEST_WEIGHTS_ROOT = "./best_flood_model_v6.pth"
RESUME_FROM       = None

OFFICIAL_TRAIN_CSV = "./flood_train_data.csv"
OFFICIAL_VAL_CSV   = "./flood_val_data.csv"

# ── Training mode ─────────────────────────────────────────────────────
# "standard"      — single train/val run on official (or fallback) split
# "ablation"      — trains 7 configs systematically, saves each
# "crossval"      — 5-fold cross-validation on official train patches
TRAINING_MODE = "standard"

# ── Ablation: which variant to train (used only when TRAINING_MODE="ablation")
# "full"               — full MANZAR v6
# "no_cbam"            — remove CBAM (direct fusion)
# "no_optical"         — SAR only (single stream, 3 channels)
# "no_opt_dropout"     — no optical dropout curriculum
# "symmetric_loss"     — FocalTversky α=β=0.5 (no FP/FN asymmetry)
# "single_stream_sar"  — baseline: single-stream UNet++ SAR only (FCN-style)
# "no_sar_ratio"       — SAR stream uses only VV+VH (no VV/VH ratio)
ABLATION_VARIANT = "full"

# ── Cross-validation ──────────────────────────────────────────────────
CV_FOLDS = 5
CV_SEED  = 42

# ── Core hyperparams ─────────────────────────────────────────────────
SEED             = 42
EPOCHS           = 250
PHYSICAL_BATCH   = 8
ACCUM_STEPS      = 4
NUM_WORKERS      = 4
LR               = 2e-4
WEIGHT_DECAY     = 1e-3
ETA_MIN          = 1e-6
PATIENCE         = 25
VAL_SMOOTHING    = 0.3
VAL_PATCH_RATIO  = 0.20

ALPHA_BCE        = 0.3
FT_ALPHA         = 0.6
FT_BETA          = 0.4
FT_GAMMA         = 4/3
OVERSAMPLE_THRESH = 0.05
OVERSAMPLE_WEIGHT = 2.0
OPT_DROP_START   = 0.25
OPT_DROP_END     = 0.05
OPT_DROP_RAMP    = 60
THRESH_SWEEP     = [0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]
IGNORE_INDEX     = -1

# ─────────────────────────────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────────────────────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(CHECKPOINT_DIR,"train.log")),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("flood_v6")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def log_hyperparameters(variant="full"):
    log.info("="*62)
    log.info("  MANZAR v6 — HYPERPARAMETER TABLE")
    log.info("="*62)
    params = {
        "TRAINING_MODE"     : TRAINING_MODE,
        "ABLATION_VARIANT"  : variant,
        "EPOCHS"            : EPOCHS,
        "PHYSICAL_BATCH"    : PHYSICAL_BATCH,
        "ACCUM_STEPS"       : ACCUM_STEPS,
        "EFFECTIVE_BATCH"   : PHYSICAL_BATCH * ACCUM_STEPS,
        "LR"                : LR,
        "WEIGHT_DECAY"      : WEIGHT_DECAY,
        "ETA_MIN"           : ETA_MIN,
        "PATIENCE"          : PATIENCE,
        "VAL_SMOOTHING_EMA" : VAL_SMOOTHING,
        "ALPHA_BCE"         : ALPHA_BCE,
        "FT_ALPHA"          : FT_ALPHA,
        "FT_BETA"           : FT_BETA,
        "FT_GAMMA"          : round(FT_GAMMA, 4),
        "OVERSAMPLE_WEIGHT" : OVERSAMPLE_WEIGHT,
        "OPT_DROP_START"    : OPT_DROP_START,
        "OPT_DROP_END"      : OPT_DROP_END,
        "OPT_DROP_RAMP"     : OPT_DROP_RAMP,
        "SEED"              : SEED,
        "SPLIT"             : "OFFICIAL" if (os.path.exists(OFFICIAL_TRAIN_CSV) and
                               os.path.exists(OFFICIAL_VAL_CSV)) else "CUSTOM_SEED42",
    }
    for k,v in params.items():
        log.info(f"  {k:<22} : {v}")
    log.info("="*62)
    # Also save to JSON for reproducibility
    with open(os.path.join(CHECKPOINT_DIR, f"hyperparams_{variant}.json"), "w") as f:
        json.dump(params, f, indent=2)
    log.info(f"  Saved: {CHECKPOINT_DIR}/hyperparams_{variant}.json")


# ─────────────────────────────────────────────────────────────────────
# 1. DATA SPLIT
# ─────────────────────────────────────────────────────────────────────
def _parse_official_ids(csv_path):
    df_csv = pd.read_csv(csv_path, header=None)
    ids = set()
    for val in df_csv.iloc[:, 0]:
        base = str(val).replace("S1Hand/","").replace("_S1Hand.tif","")
        ids.add(f"SEN1_{base}")
    return ids


def make_splits(manifest_path, val_ratio=0.20, seed=42):
    df = pd.read_csv(manifest_path)
    if "biome" not in df.columns:
        df["biome"] = df["patch_id"].apply(
            lambda x: x.split("_")[1] if "_" in x else "unknown")

    if os.path.exists(OFFICIAL_TRAIN_CSV) and os.path.exists(OFFICIAL_VAL_CSV):
        train_ids = _parse_official_ids(OFFICIAL_TRAIN_CSV)
        val_ids   = _parse_official_ids(OFFICIAL_VAL_CSV)
        train_df  = df[df["patch_id"].isin(train_ids)].reset_index(drop=True)
        val_df    = df[df["patch_id"].isin(val_ids)].reset_index(drop=True)
        unmatched = len(df) - len(train_df) - len(val_df)
        log.info("SPLIT: OFFICIAL Sen1Floods11")
        log.info(f"  Train={len(train_df)}  Val={len(val_df)}  "
                 f"Unmatched(test)={unmatched}")
        if len(train_df) > 0:
            return train_df, val_df
        log.warning("Official split matched 0 patches — falling back to custom.")

    log.info("SPLIT: CUSTOM FALLBACK (seed=42)")
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []
    for biome, group in df.groupby("biome"):
        idx = group.index.tolist()
        rng.shuffle(idx)
        n_val = max(1, int(len(idx)*val_ratio))
        val_idx.extend(idx[:n_val])
        train_idx.extend(idx[n_val:])
    return (df.loc[train_idx].reset_index(drop=True),
            df.loc[val_idx].reset_index(drop=True))


def make_kfold_splits(manifest_path, n_folds=5, seed=42):
    """
    K-fold cross-validation on the official training patches.
    Each fold holds out 1/k of the training patches as val.
    The official test set is NEVER touched.
    """
    df = pd.read_csv(manifest_path)
    if "biome" not in df.columns:
        df["biome"] = df["patch_id"].apply(
            lambda x: x.split("_")[1] if "_" in x else "unknown")

    # Use only official train patches for CV
    if os.path.exists(OFFICIAL_TRAIN_CSV):
        train_ids = _parse_official_ids(OFFICIAL_TRAIN_CSV)
        cv_df = df[df["patch_id"].isin(train_ids)].reset_index(drop=True)
        log.info(f"Cross-val pool: {len(cv_df)} official train patches")
    else:
        cv_df = df.reset_index(drop=True)
        log.info(f"Cross-val pool: {len(cv_df)} all patches (no official CSV)")

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for fold_i, (train_idx, val_idx) in enumerate(kf.split(cv_df)):
        folds.append((
            cv_df.iloc[train_idx].reset_index(drop=True),
            cv_df.iloc[val_idx].reset_index(drop=True),
            fold_i + 1
        ))
    return folds


# ─────────────────────────────────────────────────────────────────────
# 2. DATASET
# ─────────────────────────────────────────────────────────────────────
class FloodDataset(Dataset):
    def __init__(self, df, features_dir, masks_dir, is_train=True, opt_drop_prob=0.0):
        self.df            = df
        self.features_dir  = features_dir
        self.masks_dir     = masks_dir
        self.is_train      = is_train
        self.opt_drop_prob = opt_drop_prob

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        pid = self.df.iloc[idx]["patch_id"]
        x = torch.from_numpy(
            np.load(os.path.join(self.features_dir, f"{pid}.npy"))).float()
        y = torch.from_numpy(
            np.load(os.path.join(self.masks_dir, f"{pid}.npy"))).long()

        if self.is_train:
            if random.random() > 0.5:
                x = TF.hflip(x); y = TF.hflip(y.unsqueeze(0)).squeeze(0)
            if random.random() > 0.5:
                x = TF.vflip(x); y = TF.vflip(y.unsqueeze(0)).squeeze(0)
            k = random.choice([0,1,2,3])
            if k > 0:
                x = torch.rot90(x,k,[1,2]); y = torch.rot90(y,k,[0,1])
            if random.random() < self.opt_drop_prob:
                x[2:7,:,:] = 0.0
            if random.random() < 0.30:
                x[:2] = (x[:2] + torch.randn_like(x[:2])*0.02).clamp(0,1)
        return x, y


def build_train_loader(train_df, batch_size, num_workers, opt_drop_prob):
    weights = np.where(train_df["water_ratio"]>OVERSAMPLE_THRESH, OVERSAMPLE_WEIGHT, 1.0)
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    dataset = FloodDataset(train_df, FEATURES_DIR, MASKS_DIR, is_train=True, opt_drop_prob=opt_drop_prob)
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=num_workers, pin_memory=True, drop_last=True)


def build_val_loader(val_df, num_workers):
    dataset = FloodDataset(val_df, FEATURES_DIR, MASKS_DIR, is_train=False)
    return DataLoader(dataset, batch_size=16, shuffle=False,
                      num_workers=num_workers, pin_memory=True)


# ─────────────────────────────────────────────────────────────────────
# 3. ARCHITECTURE (ablation-aware)
# ─────────────────────────────────────────────────────────────────────
class CBAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg_pool     = nn.AdaptiveAvgPool2d(1)
        self.max_pool     = nn.AdaptiveMaxPool2d(1)
        self.fc           = nn.Sequential(
            nn.Linear(channels, channels//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels//reduction, channels, bias=False))
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        b,c,_,_ = x.shape
        avg_out = self.fc(self.avg_pool(x).view(b,c))
        max_out = self.fc(self.max_pool(x).view(b,c))
        x = x * torch.sigmoid(avg_out+max_out).view(b,c,1,1)
        sc = torch.cat([x.mean(1,keepdim=True), x.max(1,keepdim=True)[0]], dim=1)
        return x * torch.sigmoid(self.spatial_conv(sc))


class DualStreamUNetPP(nn.Module):
    """
    Full MANZAR v6 architecture.
    SAR  : [VV, VH, VV/VH]         — 3 channels
    Opt  : [Blue,Green,Red,NIR,NDWI] — 5 channels
    """
    def __init__(self, use_cbam=True):
        super().__init__()
        self.use_cbam    = use_cbam
        self.sar_encoder = smp.UnetPlusPlus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=3, classes=32)
        self.opt_encoder = smp.UnetPlusPlus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=5, classes=32)
        if use_cbam:
            self.cbam = CBAM(channels=64, reduction=8)
        self.fusion_head = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.15), nn.Conv2d(32, 1, kernel_size=1))

    def forward(self, x):
        sar_in = torch.cat([x[:,0:2,:,:], x[:,7:8,:,:]], dim=1)
        opt_in = x[:,2:7,:,:]
        fused  = torch.cat([self.sar_encoder(sar_in), self.opt_encoder(opt_in)], dim=1)
        if self.use_cbam:
            fused = self.cbam(fused)
        return self.fusion_head(fused)


class SingleStreamSAR(nn.Module):
    """
    Ablation baseline: single-stream SAR-only UNet++ (no optical, no CBAM).
    Equivalent to the FCN-style SAR-only baseline from Bonafilia 2020.
    Input: [VV, VH, VV/VH] — 3 channels
    """
    def __init__(self):
        super().__init__()
        self.encoder = smp.UnetPlusPlus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=3, classes=1)

    def forward(self, x):
        sar_in = torch.cat([x[:,0:2,:,:], x[:,7:8,:,:]], dim=1)
        return self.encoder(sar_in)


class SingleStreamSARnoRatio(nn.Module):
    """
    Ablation: SAR only, VV+VH only (no VV/VH ratio). 2-channel input.
    """
    def __init__(self):
        super().__init__()
        self.encoder = smp.UnetPlusPlus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=2, classes=1)

    def forward(self, x):
        return self.encoder(x[:,0:2,:,:])


def build_model(variant="full"):
    """
    Factory function — returns the correct model for each ablation variant.
    """
    if variant == "full":
        return DualStreamUNetPP(use_cbam=True)
    elif variant == "no_cbam":
        return DualStreamUNetPP(use_cbam=False)
    elif variant in ("no_optical", "single_stream_sar"):
        return SingleStreamSAR()
    elif variant == "no_sar_ratio":
        return SingleStreamSARnoRatio()
    elif variant in ("no_opt_dropout", "symmetric_loss"):
        # Same architecture as full, loss/dropout controlled in training loop
        return DualStreamUNetPP(use_cbam=True)
    else:
        raise ValueError(f"Unknown ablation variant: {variant!r}")


# ─────────────────────────────────────────────────────────────────────
# 4. LOSS
# ─────────────────────────────────────────────────────────────────────
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=FT_ALPHA, beta=FT_BETA, gamma=FT_GAMMA, smooth=1e-6):
        super().__init__()
        self.alpha=alpha; self.beta=beta; self.gamma=gamma; self.smooth=smooth

    def forward(self, logits, targets):
        probs   = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        valid   = targets != IGNORE_INDEX
        probs   = probs[valid]; targets = targets[valid].float()
        TP = (probs*targets).sum()
        FP = ((1-targets)*probs).sum()
        FN = (targets*(1-probs)).sum()
        T  = (TP+self.smooth)/(TP+self.alpha*FP+self.beta*FN+self.smooth)
        return (1-T)**self.gamma


class HybridLoss(nn.Module):
    def __init__(self, alpha_bce=ALPHA_BCE, ft_alpha=FT_ALPHA, ft_beta=FT_BETA):
        super().__init__()
        self.ft  = FocalTverskyLoss(alpha=ft_alpha, beta=ft_beta)
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.a   = alpha_bce

    def forward(self, logits, targets):
        ft_loss    = self.ft(logits, targets)
        valid_mask = (targets!=IGNORE_INDEX).float()
        bce_map    = self.bce(logits.squeeze(1), targets.float().clamp(0,1))
        bce_loss   = (bce_map*valid_mask).sum()/(valid_mask.sum()+1e-8)
        return (1-self.a)*ft_loss + self.a*bce_loss


# ─────────────────────────────────────────────────────────────────────
# 5. METRICS
# ─────────────────────────────────────────────────────────────────────
class RunningMetrics:
    def __init__(self): self.reset()

    def reset(self): self.TP=self.TN=self.FP=self.FN=0

    def update(self, probs, targets, threshold=0.5):
        p=probs.view(-1); t=targets.view(-1)
        valid=t!=IGNORE_INDEX
        p=(p[valid]>threshold).long(); t=t[valid].long()
        self.TP+=((p==1)&(t==1)).sum().item()
        self.TN+=((p==0)&(t==0)).sum().item()
        self.FP+=((p==1)&(t==0)).sum().item()
        self.FN+=((p==0)&(t==1)).sum().item()

    def compute(self):
        eps=1e-8; tp,tn,fp,fn=self.TP,self.TN,self.FP,self.FN
        prec=tp/(tp+fp+eps); rec=tp/(tp+fn+eps)
        return {"accuracy":(tp+tn)/(tp+tn+fp+fn+eps),
                "precision":prec,"recall":rec,
                "f1":2*prec*rec/(prec+rec+eps),
                "iou":tp/(tp+fp+fn+eps)}


# ─────────────────────────────────────────────────────────────────────
# 6. CHECKPOINT HELPERS
# ─────────────────────────────────────────────────────────────────────
def save_checkpoint(path, epoch, model, optimizer, scheduler,
                    best_val_iou, smoothed_val_iou, epochs_no_improve):
    torch.save({"epoch":epoch,"model_state":model.state_dict(),
                "optimizer_state":optimizer.state_dict(),
                "scheduler_state":scheduler.state_dict(),
                "best_val_iou":best_val_iou,
                "smoothed_val_iou":smoothed_val_iou,
                "epochs_no_improve":epochs_no_improve}, path)


def load_checkpoint(path, model, optimizer, scheduler, device):
    ckpt  = torch.load(path, map_location=device)
    state = {k.replace("module.",""):v for k,v in ckpt["model_state"].items()}
    model.load_state_dict(state)
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    log.info(f"Resumed epoch {ckpt['epoch']} | best val IoU {ckpt['best_val_iou']:.4f}")
    return (ckpt["epoch"], ckpt["best_val_iou"],
            ckpt.get("smoothed_val_iou", ckpt["best_val_iou"]),
            ckpt["epochs_no_improve"])


# ─────────────────────────────────────────────────────────────────────
# 7. TRAIN / VAL PASSES
# ─────────────────────────────────────────────────────────────────────
def get_opt_drop_prob(epoch, variant="full"):
    if variant == "no_opt_dropout":
        return 0.0   # ablation: no optical dropout at all
    if epoch >= OPT_DROP_RAMP: return OPT_DROP_END
    return OPT_DROP_START + (epoch/OPT_DROP_RAMP)*(OPT_DROP_END-OPT_DROP_START)


def get_criterion(variant="full"):
    if variant == "symmetric_loss":
        # ablation: symmetric FT (α=β=0.5, no FP/FN asymmetry)
        return HybridLoss(alpha_bce=ALPHA_BCE, ft_alpha=0.5, ft_beta=0.5)
    return HybridLoss(alpha_bce=ALPHA_BCE, ft_alpha=FT_ALPHA, ft_beta=FT_BETA)


def train_epoch(model, loader, criterion, optimizer, scaler, device, epoch_num):
    model.train(); running_loss=0.0; metrics=RunningMetrics(); optimizer.zero_grad()
    bar = tqdm(loader, desc=f"Train {epoch_num}", leave=False, dynamic_ncols=True)
    for step,(x,y) in enumerate(bar):
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        with torch.amp.autocast("cuda"):
            logits=model(x); loss=criterion(logits,y)/ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step+1)%ACCUM_STEPS==0 or (step+1)==len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        running_loss+=loss.item()*ACCUM_STEPS
        with torch.no_grad():
            metrics.update(torch.sigmoid(logits.detach()).squeeze(1).cpu(), y.cpu())
        bar.set_postfix(loss=f"{loss.item()*ACCUM_STEPS:.4f}")
    m=metrics.compute(); m["loss"]=running_loss/len(loader)
    return m


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval(); running_loss=0.0; all_probs,all_targets=[],[]
    for x,y in tqdm(loader, desc="Val", leave=False, dynamic_ncols=True):
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        with torch.amp.autocast("cuda"):
            logits=model(x); running_loss+=criterion(logits,y).item()
        all_probs.append(torch.sigmoid(logits).squeeze(1).cpu())
        all_targets.append(y.cpu())
    all_probs=torch.cat(all_probs); all_targets=torch.cat(all_targets)
    best_f1,best_thresh,best_metrics=0.0,0.5,None
    for t in THRESH_SWEEP:
        m=RunningMetrics(); m.update(all_probs,all_targets,threshold=t); cm=m.compute()
        if cm["f1"]>best_f1:
            best_f1,best_thresh,best_metrics=cm["f1"],t,cm
    if best_metrics is None:
        m=RunningMetrics(); m.update(all_probs,all_targets,threshold=0.5)
        best_metrics,best_thresh=m.compute(),0.5
    best_metrics["loss"]=running_loss/len(loader)
    best_metrics["thresh"]=best_thresh
    return best_metrics


# ─────────────────────────────────────────────────────────────────────
# 8. SINGLE RUN
# ─────────────────────────────────────────────────────────────────────
def train_single_run(train_df, val_df, variant="full",
                     ckpt_suffix="", resume_from=None):
    """
    One full training run for a given split and ablation variant.
    Returns best smoothed val IoU.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"\n{'='*62}")
    log.info(f"  RUN  variant={variant}  suffix={ckpt_suffix or 'none'}")
    log.info(f"  Train={len(train_df)}  Val={len(val_df)}")
    log.info(f"{'='*62}")

    best_weights_path = os.path.join(
        CHECKPOINT_DIR, f"best_{variant}{ckpt_suffix}.pth")

    model = build_model(variant)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    log.info(f"  Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=ETA_MIN)
    scaler    = torch.amp.GradScaler("cuda")
    criterion = get_criterion(variant)

    start_epoch=0; best_val_iou=0.0; smoothed_val_iou=0.0; epochs_no_improve=0
    if resume_from and os.path.exists(resume_from):
        start_epoch,best_val_iou,smoothed_val_iou,epochs_no_improve = \
            load_checkpoint(resume_from, model, optimizer, scheduler, device)

    val_loader = build_val_loader(val_df, NUM_WORKERS)

    run_csv = os.path.join(CHECKPOINT_DIR, f"log_{variant}{ckpt_suffix}.csv")
    csv_file   = open(run_csv, "w", newline="")
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(["epoch","lr","opt_drop","train_loss","train_f1","train_iou",
                          "val_loss","val_prec","val_rec","val_f1","val_iou",
                          "val_thresh","smoothed_val_iou"])

    prev_opt_drop=None; train_loader=None

    for epoch in range(start_epoch, EPOCHS):
        current_lr = optimizer.param_groups[0]["lr"]
        opt_drop   = get_opt_drop_prob(epoch, variant)

        if train_loader is None or abs(opt_drop-(prev_opt_drop or 999))>0.01:
            train_loader  = build_train_loader(train_df, PHYSICAL_BATCH, NUM_WORKERS, opt_drop)
            prev_opt_drop = opt_drop

        tm = train_epoch(model, train_loader, criterion, optimizer, scaler, device, epoch+1)
        vm = val_epoch(model, val_loader, criterion, device)
        scheduler.step()

        val_iou = vm["iou"]
        smoothed_val_iou = val_iou if smoothed_val_iou==0.0 else \
            (1-VAL_SMOOTHING)*smoothed_val_iou + VAL_SMOOTHING*val_iou

        log.info(f"  Ep{epoch+1:>3} | lr={current_lr:.1e} | "
                 f"TLoss={tm['loss']:.4f} TF1={tm['f1']:.4f} | "
                 f"VLoss={vm['loss']:.4f} VF1={vm['f1']:.4f} "
                 f"VIoU={val_iou:.4f} SmIoU={smoothed_val_iou:.4f} "
                 f"Thr={vm['thresh']:.2f} Drop={opt_drop:.2f}")
        csv_writer.writerow([epoch+1,current_lr,opt_drop,
                              tm["loss"],tm["f1"],tm["iou"],
                              vm["loss"],vm["precision"],vm["recall"],
                              vm["f1"],val_iou,vm["thresh"],smoothed_val_iou])
        csv_file.flush()

        if smoothed_val_iou > best_val_iou:
            best_val_iou=smoothed_val_iou; epochs_no_improve=0
            torch.save(model.state_dict(), best_weights_path)
            log.info(f"  >> New best SmIoU={best_val_iou:.4f} (raw={val_iou:.4f}) saved")
        else:
            epochs_no_improve+=1

        if (epoch+1)%10==0:
            save_checkpoint(
                os.path.join(CHECKPOINT_DIR, f"epoch_{epoch+1:04d}_{variant}{ckpt_suffix}.pth"),
                epoch+1, model, optimizer, scheduler,
                best_val_iou, smoothed_val_iou, epochs_no_improve)

        if epochs_no_improve >= PATIENCE:
            log.info(f"  Early stopping at epoch {epoch+1}.")
            break

    csv_file.close()
    # Copy best weights to root for easy access
    root_path = f"./best_flood_model_{variant}{ckpt_suffix}.pth"
    shutil.copy2(best_weights_path, root_path)
    log.info(f"  Best SmIoU={best_val_iou:.4f} | Saved: {root_path}")
    return best_val_iou


# ─────────────────────────────────────────────────────────────────────
# 9. ABLATION STUDY
# ─────────────────────────────────────────────────────────────────────
ABLATION_VARIANTS = [
    ("single_stream_sar", "Baseline: SAR-only single-stream UNet++ (no optical, no CBAM)"),
    ("no_sar_ratio",      "SAR-only, VV+VH only (no VV/VH ratio channel)"),
    ("no_optical",        "SAR-only with VV/VH ratio (3-channel, no optical)"),
    ("symmetric_loss",    "Dual-stream + CBAM, symmetric loss (alpha=beta=0.5)"),
    ("no_cbam",           "Dual-stream, no CBAM attention"),
    ("no_opt_dropout",    "Full dual-stream + CBAM, no optical dropout curriculum"),
    ("full",              "Full MANZAR v6 (all components)"),
]


def run_ablation_study(train_df, val_df):
    log.info("\n" + "="*62)
    log.info("  ABLATION STUDY — Training all 7 variants")
    log.info("  Each variant is trained on identical data/splits.")
    log.info("  Evaluate all with flood_official_test_v6.ipynb after.")
    log.info("="*62)
    results = {}
    for variant, description in ABLATION_VARIANTS:
        log.info(f"\n  [{variant}] {description}")
        val_iou = train_single_run(train_df, val_df, variant=variant)
        results[variant] = {"description": description, "val_iou": round(val_iou, 4)}
        log.info(f"  RESULT [{variant}]: val_iou={val_iou:.4f}")

    log.info("\n" + "="*62)
    log.info("  ABLATION SUMMARY (val IoU, ascending)")
    log.info("="*62)
    for v, r in sorted(results.items(), key=lambda x: x[1]["val_iou"]):
        log.info(f"  {v:<22} : {r['val_iou']:.4f}  — {r['description'][:45]}")
    log.info("="*62)

    with open(os.path.join(CHECKPOINT_DIR,"ablation_val_results.json"),"w") as f:
        json.dump(results, f, indent=2)
    log.info(f"  Saved: {CHECKPOINT_DIR}/ablation_val_results.json")
    return results


# ─────────────────────────────────────────────────────────────────────
# 10. CROSS-VALIDATION
# ─────────────────────────────────────────────────────────────────────
def run_cross_validation(manifest_path, variant="full"):
    log.info("\n" + "="*62)
    log.info(f"  {CV_FOLDS}-FOLD CROSS-VALIDATION  variant={variant}")
    log.info("  Official test patches are NEVER included in CV.")
    log.info("="*62)
    folds   = make_kfold_splits(manifest_path, n_folds=CV_FOLDS, seed=CV_SEED)
    fold_ious = []
    for train_df, val_df, fold_i in folds:
        log.info(f"\n  FOLD {fold_i}/{CV_FOLDS}: train={len(train_df)} val={len(val_df)}")
        val_iou = train_single_run(
            train_df, val_df,
            variant=variant,
            ckpt_suffix=f"_fold{fold_i}"
        )
        fold_ious.append(val_iou)
        log.info(f"  FOLD {fold_i} result: val_iou={val_iou:.4f}")

    mean_iou = np.mean(fold_ious)
    std_iou  = np.std(fold_ious)
    log.info("\n" + "="*62)
    log.info(f"  CV RESULT ({CV_FOLDS}-fold): {mean_iou:.4f} ± {std_iou:.4f}")
    log.info(f"  Per-fold IoUs: {[round(x,4) for x in fold_ious]}")
    log.info("="*62)
    cv_result = {"folds": fold_ious, "mean": round(mean_iou,4), "std": round(std_iou,4)}
    with open(os.path.join(CHECKPOINT_DIR, f"cv_results_{variant}.json"),"w") as f:
        json.dump(cv_result, f, indent=2)
    return cv_result


# ─────────────────────────────────────────────────────────────────────
# 11. MAIN
# ─────────────────────────────────────────────────────────────────────
def main():
    set_seed(SEED)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    torch.cuda.empty_cache(); gc.collect()

    log_hyperparameters(variant=ABLATION_VARIANT)

    if TRAINING_MODE == "standard":
        train_df, val_df = make_splits(MANIFEST_PATH, VAL_PATCH_RATIO, SEED)
        train_single_run(train_df, val_df, variant=ABLATION_VARIANT,
                         resume_from=RESUME_FROM)

    elif TRAINING_MODE == "ablation":
        train_df, val_df = make_splits(MANIFEST_PATH, VAL_PATCH_RATIO, SEED)
        run_ablation_study(train_df, val_df)

    elif TRAINING_MODE == "crossval":
        run_cross_validation(MANIFEST_PATH, variant=ABLATION_VARIANT)

    else:
        raise ValueError(f"Unknown TRAINING_MODE: {TRAINING_MODE!r}")


main()


2026-03-19 20:40:55,225 | INFO | ==============================================================
2026-03-19 20:40:55,226 | INFO |   MANZAR v6 — HYPERPARAMETER TABLE
2026-03-19 20:40:55,227 | INFO | ==============================================================
2026-03-19 20:40:55,227 | INFO |   TRAINING_MODE          : standard
2026-03-19 20:40:55,228 | INFO |   ABLATION_VARIANT       : full
2026-03-19 20:40:55,229 | INFO |   EPOCHS                 : 250
2026-03-19 20:40:55,229 | INFO |   PHYSICAL_BATCH         : 8
2026-03-19 20:40:55,230 | INFO |   ACCUM_STEPS            : 4
2026-03-19 20:40:55,231 | INFO |   EFFECTIVE_BATCH        : 32
2026-03-19 20:40:55,233 | INFO |   LR                     : 0.0002
2026-03-19 20:40:55,234 | INFO |   WEIGHT_DECAY           : 0.001
2026-03-19 20:40:55,234 | INFO |   ETA_MIN                : 1e-06
2026-03-19 20:40:55,235 | INFO |   PATIENCE               : 25
2026-03-19 20:40:55,236 | INFO |   VAL_SMOOTHING_EMA      : 0.3
2026-03-19 20:40:55,237 | INF

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

2026-03-19 20:40:56,298 | INFO | HTTP Request: HEAD https://huggingface.co/smp-hub/resnet50.imagenet/resolve/00cb74e366966d59cd9a35af57e618af9f88efe9/model.safetensors "HTTP/1.1 302 Found"
2026-03-19 20:40:56,443 | INFO | HTTP Request: GET https://huggingface.co/api/models/smp-hub/resnet50.imagenet/xet-read-token/00cb74e366966d59cd9a35af57e618af9f88efe9 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

2026-03-19 20:40:59,516 | INFO |   Params: 98,006,403
2026-03-19 20:42:20,152 | INFO |   Ep  1 | lr=2.0e-04 | TLoss=0.6296 TF1=0.4290 | VLoss=0.7273 VF1=0.6600 VIoU=0.4925 SmIoU=0.4925 Thr=0.70 Drop=0.25
2026-03-19 20:42:20,720 | INFO |   >> New best SmIoU=0.4925 (raw=0.4925) saved
2026-03-19 20:43:37,750 | INFO |   Ep  2 | lr=2.0e-04 | TLoss=0.6041 TF1=0.5119 | VLoss=0.6372 VF1=0.7603 VIoU=0.6133 SmIoU=0.5288 Thr=0.70 Drop=0.25
2026-03-19 20:43:38,600 | INFO |   >> New best SmIoU=0.5288 (raw=0.6133) saved
2026-03-19 20:44:57,859 | INFO |   Ep  3 | lr=2.0e-04 | TLoss=0.5511 TF1=0.6041 | VLoss=0.6037 VF1=0.7526 VIoU=0.6034 SmIoU=0.5511 Thr=0.70 Drop=0.24
2026-03-19 20:44:58,703 | INFO |   >> New best SmIoU=0.5511 (raw=0.6034) saved
2026-03-19 20:46:19,178 | INFO |   Ep  4 | lr=2.0e-04 | TLoss=0.5556 TF1=0.6001 | VLoss=0.6058 VF1=0.7645 VIoU=0.6188 SmIoU=0.5714 Thr=0.65 Drop=0.24
2026-03-19 20:46:20,066 | INFO |   >> New best SmIoU=0.5714 (raw=0.6188) saved
2026-03-19 20:47:40,317 | INFO